# Student 3 — Defect Detection
## Notebook 05: Integrated pipeline inference

This is the notebook to run when the three students' work is put together. It takes a **raw board
image or video**, pushes it through Student 1's preprocessing and Student 2's alignment, and runs
the trained detector on the result.

### The rule that makes it work

> The detector must be shown **exactly** the image it was trained on.

`image_pipeline.detection_input()` is the only function that produces that image, and
`00_data_preparation.ipynb` used the same function to build the training set. As long as both sides
go through it, they cannot drift apart. Calling the detector on a raw image, on a `Clean_Dataset`
image, or on Student 2's old 640 px output will produce few or no detections — that was the
original bug, not a fault in the models.

### 1. Setup

In [ ]:
import json, sys, time, warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


def find_project_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "image_pipeline.py").is_file():
            return c
    raise FileNotFoundError("Could not find image_pipeline.py")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
import image_pipeline as ip

WORK_DIR    = PROJECT_ROOT / "Student3-Defect Detection"
RESULTS_DIR = WORK_DIR / "results"
INFO        = json.loads((WORK_DIR / "dataset" / "dataset_info.json").read_text())
CLASS_NAMES = INFO["class_names"]
OUT_DIR     = WORK_DIR / "outputs"; OUT_DIR.mkdir(exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("ALIGN_TARGET :", ip.ALIGN_TARGET, "| trained at:", INFO["image_size"])
assert ip.ALIGN_TARGET == INFO["image_size"], (
    "ALIGN_TARGET no longer matches the size the models were trained at - "
    "either restore it or re-run notebooks 00-03.")
print("Classes      :", CLASS_NAMES)

### 2. Load a trained detector

Pick whichever model the benchmark in notebook 04 favoured. The wrapper hides the difference
between the Ultralytics models and the torchvision one so the rest of the notebook — and any
dashboard code built on top of it — does not have to care.

In [ ]:
MODEL_CHOICE = "yolov10"          # "yolov10" | "rtdetr" | "faster_rcnn"


class DefectDetector:
    """Uniform predict() over the three architectures. Returns boxes in ALIGNED coordinates."""

    def __init__(self, choice: str, conf: float = 0.25):
        self.choice, self.conf = choice, conf
        result_file = RESULTS_DIR / f"{choice}.json"
        if not result_file.is_file():
            raise FileNotFoundError(
                f"{result_file.name} not found - train {choice} first (notebooks 01-03).")
        self.meta = json.loads(result_file.read_text())
        weights = Path(self.meta["weights"])
        if not weights.is_file():
            raise FileNotFoundError(f"Weights missing: {weights}")

        if choice in ("yolov10", "rtdetr"):
            from ultralytics import YOLO, RTDETR
            self.model = (YOLO if choice == "yolov10" else RTDETR)(str(weights))
            self.kind = "ultralytics"
        else:
            import torch
            from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
            from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
            from torchvision.models.detection.rpn import AnchorGenerator
            self.torch = torch
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, num_classes=91)
            in_f = model.roi_heads.box_predictor.cls_score.in_features
            model.roi_heads.box_predictor = FastRCNNPredictor(in_f, len(CLASS_NAMES) + 1)
            model.rpn.anchor_generator = AnchorGenerator(
                sizes=((16,), (32,), (64,), (128,), (256,)),
                aspect_ratios=((0.5, 1.0, 2.0),) * 5)
            model.transform.min_size = (INFO["image_size"],)
            model.transform.max_size = INFO["image_size"]
            model.load_state_dict(torch.load(weights, map_location="cpu")["model"])
            self.model = model.to(self.device).eval()
            self.kind = "torchvision"

    def predict(self, aligned_bgr):
        """Returns a list of dicts: {cls, name, conf, box[x1,y1,x2,y2]} in aligned coords."""
        if self.kind == "ultralytics":
            r = self.model.predict(aligned_bgr, imgsz=INFO["image_size"], conf=self.conf,
                                   verbose=False)[0]
            return [{"cls": int(c), "name": CLASS_NAMES[int(c)], "conf": float(s),
                     "box": [float(v) for v in b]}
                    for b, c, s in zip(r.boxes.xyxy.cpu().numpy(),
                                       r.boxes.cls.cpu().numpy(),
                                       r.boxes.conf.cpu().numpy())]
        rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB)
        tensor = self.torch.from_numpy(rgb).permute(2, 0, 1).float().div(255).to(self.device)
        with self.torch.no_grad():
            out = self.model([tensor])[0]
        keep = out["scores"].cpu().numpy() >= self.conf
        return [{"cls": int(l) - 1, "name": CLASS_NAMES[int(l) - 1], "conf": float(s),
                 "box": [float(v) for v in b]}
                for b, l, s in zip(out["boxes"].cpu().numpy()[keep],
                                   out["labels"].cpu().numpy()[keep],
                                   out["scores"].cpu().numpy()[keep])]


detector = DefectDetector(MODEL_CHOICE, conf=0.25)
print(f"Loaded {detector.meta['model']} ({detector.meta['variant']})")
print(f"  test mAP@0.5 {detector.meta['test']['mAP50']:.3f} | weights {Path(detector.meta['weights']).name}")

### 3. The inspection function

`inspect_board()` is the whole integration in one place. It also returns the detections mapped back
into the **raw image's** coordinates using the inverse of the pipeline matrix, which is what a
dashboard needs when it wants to draw on the photo the operator actually took.

In [ ]:
def inspect_board(raw_bgr, conf=None):
    """
    Raw board image -> defects.

    Returns a dict with:
        aligned      : the image the detector saw (Student 2's output)
        detections   : boxes in ALIGNED coordinates
        raw_boxes    : the same boxes mapped back onto `raw_bgr`
        aligned_ok   : False if Student 2 could not find the board outline
                       (the fallback whole-frame path was used instead)
        ms           : wall-clock milliseconds for the whole chain
    """
    t0 = time.perf_counter()
    aligned, matrix, size = ip.detection_input(raw_bgr)          # S1 + S2, fallback on
    if aligned is None:
        return None
    aligned_ok = ip.quick_corner_check(ip.preprocess_image(raw_bgr))

    if conf is not None:
        detector.conf = conf
    dets = detector.predict(aligned)

    inverse = np.linalg.inv(ip.as_homography(matrix))
    raw_h, raw_w = raw_bgr.shape[:2]
    raw_boxes, _ = ip.transform_boxes([d["box"] for d in dets], inverse, raw_w, raw_h) if dets else ([], [])

    return {"aligned": aligned, "detections": dets, "raw_boxes": raw_boxes,
            "aligned_ok": aligned_ok, "size": size,
            "ms": (time.perf_counter() - t0) * 1000}


def draw(img_bgr, boxes, labels=None, colour=(0, 0, 255), thickness=2):
    """Draw boxes on a copy of an image and return it as RGB for matplotlib."""
    vis = img_bgr.copy()
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), colour, thickness)
        if labels:
            cv2.putText(vis, labels[i], (int(x1), max(int(y1) - 6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2, cv2.LINE_AA)
    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

print("inspect_board() ready.")

### 4. Inspect one raw board

A rotated board from `PCB_DATASET/rotation` is used because that is the hardest realistic input:
it arrives skewed and full resolution, exactly like a photo from the line. The left panel is the
raw input with the detections projected back onto it; the right panel is what the detector actually
saw.

In [ ]:
sample = sorted((PROJECT_ROOT / "PCB_DATASET" / "rotation" / "Missing_hole_rotation").glob("*.jpg"))[0]
raw = cv2.imread(str(sample))
res = inspect_board(raw)

print(f"input        : {sample.name}  {raw.shape[1]}x{raw.shape[0]}")
print(f"aligned to   : {res['size'][0]}x{res['size'][1]}  (board outline found: {res['aligned_ok']})")
print(f"detections   : {len(res['detections'])}   in {res['ms']:.0f} ms")
for d in res["detections"]:
    print(f"   {d['name']:<18} conf {d['conf']:.2f}  box {[round(v) for v in d['box']]}")

labels = [f"{d['name']} {d['conf']:.2f}" for d in res["detections"]]
fig, ax = plt.subplots(1, 2, figsize=(17, 7))
ax[0].imshow(draw(raw, res["raw_boxes"], labels, thickness=6))
ax[0].set_title(f"raw input, detections projected back ({sample.name})"); ax[0].axis("off")
ax[1].imshow(draw(res["aligned"], [d["box"] for d in res["detections"]], labels))
ax[1].set_title("what the detector saw (Student 2 output)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

### 5. Batch inspection

The batch path a production line would use: a folder of boards in, one row per board out. This also
gives the honest end-to-end throughput, including Student 1 and Student 2 — the alignment stage is
usually the slowest part on CPU, not the network.

In [ ]:
ROTATION_ROOT = PROJECT_ROOT / "PCB_DATASET" / "rotation"
BATCH_DIR = ROTATION_ROOT / "Short_rotation"
if not BATCH_DIR.is_dir():                       # fall back to whichever folder exists
    BATCH_DIR = next(d for d in sorted(ROTATION_ROOT.iterdir()) if d.is_dir())
BATCH_N   = 12
print("batch folder:", BATCH_DIR.name)

rows, timings = [], []
for path in sorted(BATCH_DIR.glob("*.jpg"))[:BATCH_N]:
    raw = cv2.imread(str(path))
    r = inspect_board(raw)
    if r is None:
        rows.append((path.name, "PIPELINE FAILED", 0, 0.0)); continue
    names = ", ".join(sorted({d["name"] for d in r["detections"]})) or "-"
    rows.append((path.name, names, len(r["detections"]), r["ms"]))
    timings.append(r["ms"])

print(f"{'board':<30}{'defect types':<34}{'n':>4}{'ms':>9}")
for name, types, n, ms in rows:
    print(f"{name:<30}{types:<34}{n:>4}{ms:>9.0f}")

if timings:
    print(f"\nend-to-end (S1 + S2 + detector): {np.mean(timings):.0f} ms/board "
          f"-> {1000/np.mean(timings):.1f} boards per second")
    print(f"boards with at least one defect : {sum(1 for r in rows if r[2] > 0)}/{len(rows)}")

### 6. Video inspection

`Student1-Image Acquisition & Pre-processing/PCB_Conveyor.mp4` is run frame by frame through the
same function and written out annotated. Alignment is attempted on every frame; frames where the
board is only partly in view fall back to the whole-frame path, which is why `aligned_ok` is worth
logging.

In [ ]:
RUN_VIDEO  = False        # set True to process the conveyor clip (slow on CPU)
VIDEO_IN   = PROJECT_ROOT / "Student1-Image Acquisition & Pre-processing" / "PCB_Conveyor.mp4"
VIDEO_OUT  = OUT_DIR / "conveyor_inspected.mp4"
MAX_FRAMES = 120

if RUN_VIDEO and VIDEO_IN.is_file():
    cap = cv2.VideoCapture(str(VIDEO_IN))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    writer, n, hits = None, 0, 0
    while n < MAX_FRAMES:
        ok, frame = cap.read()
        if not ok:
            break
        r = inspect_board(frame)
        if r is None:
            continue
        vis = cv2.cvtColor(draw(r["aligned"], [d["box"] for d in r["detections"]],
                                [f"{d['name']} {d['conf']:.2f}" for d in r["detections"]]),
                           cv2.COLOR_RGB2BGR)
        if writer is None:
            # the aligned size can vary slightly frame to frame, so the first frame
            # fixes the output size and every later frame is resized to match it
            out_h, out_w = vis.shape[:2]
            writer = cv2.VideoWriter(str(VIDEO_OUT), cv2.VideoWriter_fourcc(*"mp4v"),
                                     fps, (out_w, out_h))
        elif vis.shape[:2] != (out_h, out_w):
            vis = cv2.resize(vis, (out_w, out_h))
        writer.write(vis)
        hits += len(r["detections"]); n += 1
    cap.release()
    if writer:
        writer.release()
    print(f"{n} frames processed, {hits} detections -> {VIDEO_OUT}")
else:
    print("RUN_VIDEO is False (or the clip is missing) - skipped.")

### 7. If it still detects nothing — checklist

Work down this list; each item was a real failure mode in this project.

| Symptom | Cause | Fix |
|---|---|---|
| Zero detections on every board | The detector was given a raw / `Clean_Dataset` / old 640 px image | Always call `ip.detection_input()`; never `model.predict(raw)` |
| Zero detections after changing Student 2 | `ALIGN_TARGET` no longer matches `dataset_info.json` | The assertion in Section 1 catches this — re-run notebooks 00–03 |
| Detections in the wrong place | Boxes drawn on the raw image without the inverse homography | Use `res["raw_boxes"]`, not `res["detections"]` |
| Very few detections, low confidence | Confidence threshold too high for small defects | `inspect_board(raw, conf=0.10)` and re-check |
| Works on originals, fails on rotated boards | Alignment failed and the fallback path was used | Check `res["aligned_ok"]`; inspect `board_corners_threshold` on that board |
| Good mAP in notebook 01 but nothing here | Different model loaded than the one benchmarked | Check `detector.meta["weights"]` points at the run you evaluated |

The cell below re-runs the two checks that matter most, so this notebook fails loudly rather than
quietly returning empty lists.

In [ ]:
checks = []
checks.append(("ALIGN_TARGET matches training size",
               ip.ALIGN_TARGET == INFO["image_size"]))

train_img = sorted((Path(INFO["paths"]["yolo_root"]) / "images" / "train").glob("*.jpg"))[0]
t_h, t_w = cv2.imread(str(train_img)).shape[:2]
checks.append(("training images are aligned boards at the align target",
               max(t_w, t_h) == ip.ALIGN_TARGET))

probe = inspect_board(cv2.imread(str(sample)), conf=0.05)
checks.append(("pipeline returns an aligned image", probe is not None and probe["aligned"] is not None))
checks.append(("detector produces at least one candidate at conf 0.05",
               probe is not None and len(probe["detections"]) > 0))

for name, ok in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
detector.conf = 0.25
if all(ok for _, ok in checks):
    print("\nIntegration is wired correctly.")